In [0]:
import logging
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.functions import vector_to_array  # Requerido para manejar probabilidades en Spark 3+

# =========================================================================
# 1. CONFIGURACIÓN
# =========================================================================
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] - %(message)s')
logger = logging.getLogger(__name__)

# =========================================================================
# 2. PIPELINE DE MACHINE LEARNING EN SPARK (100% DISTRIBUIDO)
# =========================================================================
class SumerSportsSparkPlayPredictor:
    """
    Pipeline de Machine Learning Real usando PySpark MLlib.
    Procesa arrays de Spark nativamente y ejecuta el entrenamiento de forma distribuida.
    """
    def __init__(self, spark_session: SparkSession, feature_table: str):
        self.spark = spark_session
        self.feature_table = feature_table
        
        # Modelo distribuido de Spark ML
        self.rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100, maxDepth=5)
        self.model = None
        self._is_trained = False

    def get_data(self):
        logger.info("Extrayendo Feature Store (OBT) nativamente con Spark...")
        # NOTA: Eliminamos .toPandas() para que la data nunca salga de los nodos de Spark
        return self.spark.read.table(self.feature_table)

    def build_features(self, df):
        """
        FEATURE ENGINEERING DISTRIBUIDO:
        En lugar de usar Numpy y Pandas de nodo único, usamos funciones nativas 
        de Spark SQL que procesan los arreglos en paralelo.
        """
        logger.info("Construyendo matriz de características (Feature Extraction en Spark)...")
        
        # 1. Extraemos los sub-arreglos de velocidad y aceleración del array de structs
        df_features = df.withColumn("vel_array", F.col("telemetry_history_array.speed_mph")) \
                        .withColumn("acc_array", F.col("telemetry_history_array.acceleration_m_s2"))
        
        # 2. Calculamos estadísticas directamente sobre los arreglos (sin explotar filas)
        df_features = df_features.withColumn("mean_vel", 
                                            F.expr("aggregate(vel_array, 0D, (acc, x) -> acc + x) / size(vel_array)")) \
                                .withColumn("max_vel", F.array_max("vel_array")) \
                                .withColumn("min_vel", F.array_min("vel_array")) \
                                .withColumn("mean_acc", 
                                            F.expr("aggregate(acc_array, 0D, (acc, x) -> acc + x) / size(acc_array)")) \
                                .withColumn("max_acc", F.array_max("acc_array")) \
                                .withColumn("min_acc", F.array_min("acc_array"))
        
        # Rellenamos nulos con 0.0 en caso de telemetrías vacías
        feature_cols = ["mean_vel", "max_vel", "min_vel", "mean_acc", "max_acc", "min_acc"]
        df_features = df_features.fillna(0.0, subset=feature_cols)
        
        return df_features

    def train_dummy_model(self, df_features):
        """
        Entrenamos el modelo usando el ecosistema de Spark ML con Pipeline y VectorAssembler.
        """
        logger.info("Entrenando algoritmo Random Forest de Spark (Simulación de pesos)...")
        
        # Simulamos etiquetas (0 o 1) de forma distribuida
        df_labeled = df_features.withColumn("label", F.when(F.rand() > 0.5, 1).otherwise(0))
        
        # FIX para tablas con 1 sola fila: forzar balanceo de clases duplicando el registro
        count = df_labeled.count()
        if count == 1:
            row0 = df_labeled.withColumn("label", F.lit(0))
            row1 = df_labeled.withColumn("label", F.lit(1))
            df_labeled = row0.union(row1)
        
        # Configuración de las variables predictoras para Spark ML
        feature_cols = ["mean_vel", "max_vel", "min_vel", "mean_acc", "max_acc", "min_acc"]
        assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
        
        # Ensamblamos el Pipeline
        pipeline = Pipeline(stages=[assembler, self.rf])
        
        # Entrenamos de forma distribuida en el cluster
        self.model = pipeline.fit(df_labeled)
        self._is_trained = True

    def predict(self, df):
        """
        Inferencia distribuida nativa. Capaz de procesar miles de millones de filas.
        """
        # 1. Ingeniería de características
        df_features = self.build_features(df)
        
        # 2. Entrenamiento del modelo simulado (si no está entrenado)
        if not self._is_trained:
            self.train_dummy_model(df_features)
            
        logger.info("Ejecutando inferencia de Machine Learning con Spark...")
        
        # 3. El pipeline ejecuta VectorAssembler + RandomForest automáticamente
        predictions = self.model.transform(df_features)
        
        # 4. En Spark ML, la predicción probabilística devuelve un vector [P(Clase 0), P(Clase 1)]
        # Extraemos la probabilidad de la clase exitosa (Clase 1) usando vector_to_array
        predictions = predictions.withColumn("prob_array", vector_to_array("probability")) \
                                 .withColumn("ml_win_probability_pct", F.round(F.col("prob_array")[1] * 100, 2))
        
        # 5. Clasificación distribuida usando F.when (Equivalente de Spark a np.select)
        df_final = predictions.withColumn(
            "estrategia",
            F.when(F.col("ml_win_probability_pct") >= 65.0, F.lit("Alta Confianza"))
             .when(F.col("ml_win_probability_pct") >= 40.0, F.lit("50/50 Incertidumbre"))
             .otherwise(F.lit("Baja Confianza"))
        )
        
        # Devolvemos solo las columnas requeridas para mantener consistencia
        return df_final.select("game_id", "play_id", "ml_win_probability_pct", "estrategia")

# =========================================================================
# 3. EJECUCIÓN EN DATABRICKS
# =========================================================================
if __name__ == "__main__":
    TABLE_GOLD = "dev_nfl_telemetry_kafka.dev_nfl_telemetry_kafka_gold.gold_play_trajectory"
    
    try:
        # Inicializamos el predictor
        ml_pipeline = SumerSportsSparkPlayPredictor(spark_session=spark, feature_table=TABLE_GOLD)
        
        # 1. Obtener datos (Mantiene la estructura de Spark DataFrame)
        df_raw = ml_pipeline.get_data()
        
        # 2. Ejecutar Inferencia distribuidamente
        df_predictions = ml_pipeline.predict(df_raw)
        
        logger.info("✅ Predicciones distribuidas generadas con éxito.")
        
        # Usamos display() que es la forma nativa y elegante de Databricks para visualizar DataFrames
        display(df_predictions.limit(10))
        
    except Exception as e:
        logger.critical(f"Fallo en el pipeline de ML de Spark: {str(e)}")